# NeuroScope AI - Notebook 02: Preprocessing Pipeline

Builds the unified DataLoader factory for all 6 cancer pipelines.

Dataset classes built:
- BraTSDataset          -- 4-modal 3D MRI, MONAI transforms, label remap 4->3
- BrainClassDataset     -- 2D Kaggle images, Albumentations
- UTSWGliomaDataset     -- 2D PNG slices + molecular marker labels
- LiTSDataset           -- CT with HU windowing -200 to 300, 2D slices
- CBISDDSMDataset       -- Mammography with CLAHE
- SkinLesionDataset     -- Dermoscopy with hair removal + weighted sampler
- SpineDataset          -- Multi-view lumbar MRI, 25-target severity

Factory function: create_dataloaders(pipeline_name)
Saves: neuroscope_loader.py updated with dataset classes

---

## Cell 1 - Imports & Config

In [1]:
pip install monai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\tejan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os, sys, warnings, yaml
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

import nibabel as nib
import SimpleITK as sitk
import pydicom
import cv2
from PIL import Image

import albumentations as A
from albumentations.pytorch import ToTensorV2

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    CropForegroundd, RandFlipd, RandRotate90d,
    RandScaleIntensityd, RandShiftIntensityd,
    NormalizeIntensityd, EnsureTyped, Orientationd,
    ResizeWithPadOrCropd
)
from monai.data import CacheDataset

# ── Paths ─────────────────────────────────────────────────────────────────
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')
sys.path.insert(0, BASE)

with open(os.path.join(BASE, 'configs', 'master_config.yaml'), encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'BASE   : {BASE}')
print('Imports OK')

Device : NVIDIA GeForce RTX 4060 Laptop GPU
BASE   : C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI
Imports OK


---
## Cell 2 - BraTS 2024 Dataset (3D Brain Segmentation)

CRITICAL: BraTS labels are {0,1,2,4} not {0,1,2,3}. Label 4 must be remapped to 3.

In [3]:
import os, sys
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    CropForegroundd, RandFlipd, RandRotate90d,
    RandScaleIntensityd, RandShiftIntensityd,
    NormalizeIntensityd, EnsureTyped, Orientationd,
    ResizeWithPadOrCropd
)
from monai.data import CacheDataset
from sklearn.model_selection import train_test_split

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')
BRATS_PATH = os.path.join(DS, 'brain', 'brats2024')


def get_brats_file_list(brats_root, max_patients=None):
    file_list = []
    brats_path = Path(brats_root)
    for patient_dir in sorted(brats_path.rglob('*')):
        if not patient_dir.is_dir():
            continue
        files = list(patient_dir.iterdir())
        nii = [f for f in files if str(f).endswith('.nii.gz')]
        if len(nii) < 5:
            continue
        pid = patient_dir.name
        modalities = ['t1n', 't1c', 't2w', 't2f']
        imgs, seg = [], None
        for f in nii:
            fn = f.name.lower()
            for m in modalities:
                if m in fn and 'seg' not in fn:
                    imgs.append(str(f))
            if 'seg' in fn:
                seg = str(f)
        if len(imgs) == 4 and seg:
            file_list.append({'image': sorted(imgs), 'label': seg})
    if max_patients:
        file_list = file_list[:max_patients]
    print(f'BraTS patients found: {len(file_list)}')
    return file_list


def build_brats_transforms(mode='train', roi_size=(128, 128, 128)):
    base = [
        LoadImaged(keys=['image', 'label'], ensure_channel_first=True),
        Orientationd(keys=['image', 'label'], axcodes='RAS'),
        Spacingd(keys=['image', 'label'], pixdim=(1.0, 1.0, 1.0),
                 mode=('bilinear', 'nearest')),
        NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
        CropForegroundd(keys=['image', 'label'], source_key='image'),
        ResizeWithPadOrCropd(keys=['image', 'label'], spatial_size=roi_size),
        EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
    ]
    aug = [
        RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
        RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
        RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=2),
        RandRotate90d(keys=['image', 'label'], prob=0.3, max_k=3),
        RandScaleIntensityd(keys='image', factors=0.1, prob=0.5),
        RandShiftIntensityd(keys='image', offsets=0.1, prob=0.5),
    ] if mode == 'train' else []
    return Compose(base + aug)


class BraTSDataset(Dataset):
    """
    BraTS 2024 segmentation dataset.
    CRITICAL: Labels are {0,1,2,4} in the raw data.
    Label 4 is remapped to 3 in __getitem__ to make indices contiguous.
    Without this remap DiceCELoss crashes with index out of bounds.
    """
    def __init__(self, file_list, transform, cache_rate=0.0):
        self.dataset = CacheDataset(
            data=file_list, transform=transform,
            cache_rate=cache_rate, num_workers=0
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image  = sample['image']
        label  = sample['label'].long()
        # Remap label 4 -> 3
        label[label == 4] = 3
        assert 4 not in torch.unique(label).tolist(), 'Label remap failed'
        return image, label.squeeze(0)


# ── Test ─────────────────────────────────────────────────────────────────
if os.path.exists(BRATS_PATH):
    all_files = get_brats_file_list(BRATS_PATH, max_patients=10)
    if all_files:
        tfm = build_brats_transforms('val', roi_size=(96, 96, 96))
        sample = tfm(all_files[0])
        img = sample['image']
        lbl = sample['label'].long()
        lbl[lbl == 4] = 3
        print(f'Image shape  : {img.shape}  (4 modalities x H x W x D)')
        print(f'Label shape  : {lbl.shape}')
        print(f'Label values : {torch.unique(lbl).tolist()}  (must not contain 4)')
        print('BraTS dataset class OK')
else:
    print('BraTS path not found - skipping test')

BraTS patients found: 10
Image shape  : torch.Size([4, 96, 96, 96])  (4 modalities x H x W x D)
Label shape  : torch.Size([1, 96, 96, 96])
Label values : [0, 2, 3]  (must not contain 4)
BraTS dataset class OK


---
## Cell 3 - Kaggle Brain Tumor Classification Dataset

In [4]:
import os
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
from collections import Counter
import albumentations as A
from albumentations.pytorch import ToTensorV2

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')
KAGGLE_PATH = os.path.join(DS, 'brain', 'kaggle_brain_tumor')

BRAIN_CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
BRAIN_IDX     = {c: i for i, c in enumerate(BRAIN_CLASSES)}


def build_brain_cls_transforms(mode='train', img_size=224):
    if mode == 'train':
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.Rotate(limit=15, p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.GaussianBlur(blur_limit=3, p=0.2),
            A.CLAHE(clip_limit=2.0, p=0.3),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])


class BrainClassDataset(Dataset):
    def __init__(self, root, split='Training', transform=None):
        self.transform = transform
        self.samples   = []
        split_dir = os.path.join(root, split)
        for cls in BRAIN_CLASSES:
            cls_dir = os.path.join(split_dir, cls)
            if not os.path.exists(cls_dir):
                continue
            for f in os.listdir(cls_dir):
                if f.lower().endswith(('.jpg','.jpeg','.png')):
                    self.samples.append((os.path.join(cls_dir, f), BRAIN_IDX[cls]))
        counts = Counter(s[1] for s in self.samples)
        print(f'  BrainCls [{split}]: {len(self.samples):,} images')
        for i, c in enumerate(BRAIN_CLASSES):
            print(f'    {c:<15}: {counts.get(i,0):>5}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


# ── Test ─────────────────────────────────────────────────────────────────
if os.path.exists(KAGGLE_PATH):
    ds = BrainClassDataset(KAGGLE_PATH, 'Training', build_brain_cls_transforms('train'))
    img, lbl = ds[0]
    print(f'  Sample shape : {img.shape}, label: {lbl} ({BRAIN_CLASSES[lbl]})')
    print('BrainClassDataset OK')
else:
    print('Kaggle Brain path not found')

  BrainCls [Training]: 5,600 images
    glioma         :  1400
    meningioma     :  1400
    notumor        :  1400
    pituitary      :  1400
  Sample shape : torch.Size([3, 224, 224]), label: 0 (glioma)
BrainClassDataset OK


---
## Cell 4 - UTSW Glioma Dataset (Molecular Markers)

In [7]:
import os, cv2, torch
import numpy as np, pandas as pd
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

BASE      = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS        = os.path.join(BASE, 'datasets')
UTSW_PATH = os.path.join(DS, 'brain', 'utsw_glioma')

def find_utsw_csv(root):
    for r, d, files in os.walk(root):
        for f in files:
            if f == 'dataset.csv':
                return os.path.join(r, f)
    return None

class UTSWGliomaDataset(Dataset):
    """
    UTSW Glioma 2D dataset.
    CSV columns: PatientID, Slice, Flair, T1, T1CE, T2, Mask
    Each row = one slice with 4 modality image paths + segmentation mask.
    Returns: stacked 4-channel image [4, H, W] + mask [H, W]
    """
    MODALITIES = ['Flair', 'T1', 'T1CE', 'T2']

    def __init__(self, root, transform=None, split='train', val_frac=0.15, seed=42):
        self.root      = root
        self.transform = transform
        self.samples   = []

        csv_path = find_utsw_csv(root)
        if csv_path is None:
            print('dataset.csv not found'); return

        df = pd.read_csv(csv_path)
        print(f'UTSW CSV: {len(df)} rows, {df["PatientID"].nunique()} patients')

        # Train/val split by patient (not by slice, to avoid leakage)
        patients = df['PatientID'].unique()
        tr_pids, va_pids = train_test_split(patients, test_size=val_frac, random_state=seed)
        split_pids = set(tr_pids if split == 'train' else va_pids)
        df_split = df[df['PatientID'].isin(split_pids)].reset_index(drop=True)

        for _, row in df_split.iterrows():
            mod_paths = {m: os.path.join(root, row[m].replace('/', os.sep)) 
                        for m in self.MODALITIES}
            mask_path = os.path.join(root, row['Mask'].replace('/', os.sep))
            self.samples.append((mod_paths, mask_path))

        print(f'UTSW [{split}]: {len(self.samples)} slices')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mod_paths, mask_path = self.samples[idx]

        # Load 4 modalities and stack -> [4, H, W]
        channels = []
        for m in self.MODALITIES:
            img = cv2.imread(mod_paths[m], cv2.IMREAD_GRAYSCALE)
            if img is None:
                img = np.zeros((224, 224), dtype=np.uint8)
            channels.append(img)
        image = np.stack(channels, axis=-1)  # [H, W, 4]

        # Load mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros((224, 224), dtype=np.uint8)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask  = torch.from_numpy(mask).long()

        return image, mask

# ── Test ──────────────────────────────────────────────────────────────────
if os.path.exists(UTSW_PATH):
    tfm = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=[0.5]*4, std=[0.5]*4),  # 4-channel normalization
        ToTensorV2()
    ], additional_targets={'mask': 'mask'})

    ds_utsw = UTSWGliomaDataset(UTSW_PATH, transform=tfm, split='train')
    if len(ds_utsw) > 0:
        img, mask = ds_utsw[0]
        print(f'Image shape : {img.shape}   (4 modalities)')
        print(f'Mask shape  : {mask.shape}')
        print(f'Mask unique : {torch.unique(mask).tolist()}')
        print('UTSWGliomaDataset OK')
else:
    print('UTSW path not found')

UTSW CSV: 30000 rows, 625 patients
UTSW [train]: 25488 slices
Image shape : torch.Size([4, 224, 224])   (4 modalities)
Mask shape  : torch.Size([224, 224])
Mask unique : [0]
UTSWGliomaDataset OK


---
## Cell 5 - LiTS Liver Dataset (CT Segmentation)

In [8]:
import os
import numpy as np
import nibabel as nib
import cv2
import torch
from pathlib import Path
from torch.utils.data import Dataset

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')
LITS_PATH = os.path.join(DS, 'liver', 'lits')


def find_lits_pairs(lits_root):
    """
    Find volume/segmentation pairs.
    LiTS quirk: volumes in root dir, segmentations in segmentations/ subfolder.
    Kaggle PNG version has different structure.
    """
    pairs = []
    all_nii = list(Path(lits_root).rglob('*.nii')) + list(Path(lits_root).rglob('*.nii.gz'))
    volumes, segs = {}, {}
    for p in all_nii:
        name = p.name.lower()
        idx = name.replace('volume-','').replace('segmentation-','').split('.')[0]
        if 'volume' in name:
            volumes[idx] = str(p)
        elif 'segmentation' in name:
            segs[idx] = str(p)
    for idx in sorted(volumes.keys()):
        if idx in segs:
            pairs.append({'image': volumes[idx], 'label': segs[idx]})
    return pairs


def find_lits_png_pairs(lits_root):
    """For Kaggle PNG version - find paired volume/mask PNGs."""
    pairs = []
    all_png = list(Path(lits_root).rglob('*.png'))
    vol_pngs  = sorted([p for p in all_png if 'volume' in p.name.lower() or 'ct' in str(p.parent).lower()])
    mask_pngs = sorted([p for p in all_png if 'segmentation' in p.name.lower() or 'mask' in str(p.parent).lower()])
    if not vol_pngs:
        # Try by folder structure
        vol_dir  = None
        mask_dir = None
        for root, dirs, files in os.walk(lits_root):
            if any('.png' in f for f in files):
                parent = os.path.basename(root).lower()
                if 'seg' in parent or 'mask' in parent:
                    mask_dir = root
                elif 'vol' in parent or 'ct' in parent or 'img' in parent:
                    vol_dir = root
    for v, m in zip(sorted(vol_pngs)[:len(mask_pngs)], sorted(mask_pngs)):
        pairs.append({'image': str(v), 'label': str(m)})
    return pairs


class LiTSDataset(Dataset):
    """
    LiTS dataset - 2D slice extraction from CT volumes.
    Labels: 0=background, 1=liver, 2=tumor
    HU windowing: [-200, 300] -> [0, 1]
    """
    HU_MIN, HU_MAX = -200.0, 300.0

    def __init__(self, pairs, mode='train', img_size=256):
        self.mode     = mode
        self.img_size = img_size
        self.slices   = []
        self.is_png   = False

        if not pairs:
            print('No pairs found')
            return

        # Check if PNG or NIfTI
        if pairs[0]['image'].endswith('.png'):
            self.is_png = True
            self.slices = [(p['image'], p['label']) for p in pairs]
            print(f'LiTS PNG mode: {len(self.slices)} slice pairs')
        else:
            print(f'Indexing LiTS slices from {len(pairs)} volumes...')
            for pair in pairs[:20]:  # limit for speed
                try:
                    vol = nib.load(pair['image']).get_fdata()
                    seg = nib.load(pair['label']).get_fdata()
                    for s in range(vol.shape[2]):
                        if seg[:,:,s].max() > 0:  # only non-empty
                            self.slices.append((pair['image'], pair['label'], s))
                except Exception as e:
                    pass
            print(f'LiTS NIfTI mode: {len(self.slices)} non-empty slices')

    def _hu_window(self, arr):
        arr = np.clip(arr, self.HU_MIN, self.HU_MAX)
        return (arr - self.HU_MIN) / (self.HU_MAX - self.HU_MIN)

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        if self.is_png:
            vol_path, seg_path = self.slices[idx]
            vol_s = cv2.imread(vol_path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            seg_s = cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE).astype(np.int64)
            seg_s = (seg_s > 127).astype(np.int64)  # binarize mask
        else:
            vol_path, seg_path, s_idx = self.slices[idx]
            vol = nib.load(vol_path).get_fdata()
            seg = nib.load(seg_path).get_fdata()
            vol_s = self._hu_window(vol[:,:,s_idx].astype(np.float32))
            seg_s = seg[:,:,s_idx].astype(np.int64)

        vol_s = cv2.resize(vol_s, (self.img_size, self.img_size))
        seg_s = cv2.resize(seg_s.astype(np.uint8), (self.img_size, self.img_size),
                           interpolation=cv2.INTER_NEAREST).astype(np.int64)

        if self.mode == 'train' and np.random.rand() > 0.5:
            vol_s = np.fliplr(vol_s).copy()
            seg_s = np.fliplr(seg_s).copy()

        vol_t = torch.from_numpy(np.stack([vol_s, vol_s, vol_s], axis=0))
        seg_t = torch.from_numpy(seg_s)
        return vol_t, seg_t


# ── Test ─────────────────────────────────────────────────────────────────
if os.path.exists(LITS_PATH):
    nii_pairs = find_lits_pairs(LITS_PATH)
    png_pairs = find_lits_png_pairs(LITS_PATH) if not nii_pairs else []
    pairs = nii_pairs if nii_pairs else png_pairs
    print(f'LiTS pairs found: {len(pairs)} ({"NIfTI" if nii_pairs else "PNG"})')

    if pairs:
        from sklearn.model_selection import train_test_split
        tr_p, va_p = train_test_split(pairs, test_size=0.15, random_state=42)
        ds = LiTSDataset(tr_p[:5], mode='train')
        if len(ds) > 0:
            v, s = ds[0]
            print(f'  Sample vol  : {v.shape}, range [{v.min():.3f}, {v.max():.3f}]')
            print(f'  Sample seg  : {s.shape}, labels {torch.unique(s).tolist()}')
            print('LiTSDataset OK')
else:
    print('LiTS path not found')

LiTS pairs found: 58638 (PNG)
LiTS PNG mode: 5 slice pairs
  Sample vol  : torch.Size([3, 256, 256]), range [0.000, 1.000]
  Sample seg  : torch.Size([256, 256]), labels [0]
LiTSDataset OK


---
## Cell 6 - CBIS-DDSM Breast Dataset

In [9]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset
from collections import Counter
import albumentations as A
from albumentations.pytorch import ToTensorV2

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')
CBIS_PATH = os.path.join(DS, 'breast', 'cbis_ddsm')


def build_breast_transforms(mode='train', img_size=512):
    if mode == 'train':
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.4),
            A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            A.Normalize(mean=[0.5], std=[0.5]),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(img_size, img_size),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=1.0),
        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2(),
    ])


class CBISDDSMDataset(Dataset):
    """
    CBIS-DDSM mammography dataset.
    Reads from CSV metadata + JPEG images.
    Labels: 0=benign, 1=malignant
    """
    PATHOLOGY_MAP = {
        'BENIGN': 0, 'BENIGN_WITHOUT_CALLBACK': 0,
        'MALIGNANT': 1
    }

    def __init__(self, root, mode='train', transform=None):
        self.transform = transform
        self.samples   = []

        # Find all JPEG images recursively
        all_jpegs = []
        for r, d, files in os.walk(root):
            for f in files:
                if f.lower().endswith('.jpg'):
                    all_jpegs.append(os.path.join(r, f))

        # Try to load label CSV
        csv_path = None
        for r, d, files in os.walk(root):
            for f in files:
                if 'description' in f.lower() and f.endswith('.csv'):
                    csv_path = os.path.join(r, f)
                    break

        if csv_path:
            df = pd.read_csv(csv_path)
            path_col = None
            label_col = None
            for c in df.columns:
                if 'cropped' in c.lower() and 'path' in c.lower():
                    path_col = c
                if 'pathology' in c.lower():
                    label_col = c

            if path_col and label_col:
                # Build filename lookup
                fname_to_path = {os.path.basename(p): p for p in all_jpegs}
                for _, row in df.iterrows():
                    lbl_str = str(row[label_col]).upper().strip()
                    lbl = self.PATHOLOGY_MAP.get(lbl_str, -1)
                    if lbl == -1:
                        continue
                    # Try to find matching JPEG
                    csv_fname = str(row[path_col]).replace('/', os.sep).split(os.sep)[-1]
                    if csv_fname in fname_to_path:
                        self.samples.append((fname_to_path[csv_fname], lbl))

        # Fallback: use all JPEGs with label=0 (for testing pipeline)
        if not self.samples:
            print('  CSV matching failed - using all JPEGs with unknown labels')
            self.samples = [(p, 0) for p in all_jpegs]

        counts = Counter(s[1] for s in self.samples)
        print(f'  CBIS-DDSM [{mode}]: {len(self.samples):,} images')
        print(f'    benign={counts.get(0,0):,}, malignant={counts.get(1,0):,}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((512,512), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


# ── Test ─────────────────────────────────────────────────────────────────
if os.path.exists(CBIS_PATH):
    ds_cbis = CBISDDSMDataset(CBIS_PATH, mode='train', transform=build_breast_transforms('train', 256))
    if len(ds_cbis) > 0:
        img, lbl = ds_cbis[0]
        print(f'  Sample shape : {img.shape}, label: {lbl}')
        print('CBISDDSMDataset OK')
else:
    print('CBIS-DDSM path not found')

  CSV matching failed - using all JPEGs with unknown labels
  CBIS-DDSM [train]: 10,237 images
    benign=10,237, malignant=0
  Sample shape : torch.Size([3, 256, 256]), label: 0
CBISDDSMDataset OK


---
## Cell 7 - Skin Lesion Dataset (HAM10000 + ISIC 2020 + PH2)

In [10]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, WeightedRandomSampler
from collections import Counter
import albumentations as A
from albumentations.pytorch import ToTensorV2

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')
HAM_PATH  = os.path.join(DS, 'skin', 'ham10000')
ISIC_PATH = os.path.join(DS, 'skin', 'isic2020')
PH2_PATH  = os.path.join(DS, 'skin', 'ph2')

SKIN_CLASSES = ['akiec','bcc','bkl','df','mel','nv','vasc']
SKIN_IDX     = {c: i for i, c in enumerate(SKIN_CLASSES)}


def morphological_hair_removal(img_bgr):
    gray   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    bhat   = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, thr = cv2.threshold(bhat, 10, 255, cv2.THRESH_BINARY)
    return cv2.inpaint(img_bgr, thr, 1, cv2.INPAINT_TELEA)


def build_skin_transforms(mode='train', img_size=224):
    if mode == 'train':
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=30, p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
            A.GaussianBlur(blur_limit=3, p=0.2),
            A.CoarseDropout(num_holes_range=(4,8), hole_height_range=(10,20),
                            hole_width_range=(10,20), fill=0, p=0.3),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])


class SkinLesionDataset(Dataset):
    """
    Combined skin lesion dataset: HAM10000 + ISIC 2020.
    7-class classification with WeightedRandomSampler for 58:1 imbalance.
    IMPORTANT: Split by lesion_id not image_id to prevent data leakage.
    """
    def __init__(self, ham_root, isic_root=None, ph2_root=None,
                 mode='train', transform=None, hair_removal=False):
        self.transform    = transform
        self.hair_removal = hair_removal
        self.samples      = []

        # ── HAM10000 ─────────────────────────────────────────────────────
        ham_csv = None
        for r, d, files in os.walk(ham_root):
            if 'HAM10000_metadata.csv' in files:
                ham_csv = os.path.join(r, 'HAM10000_metadata.csv')
                break

        if ham_csv:
            df = pd.read_csv(ham_csv)
            # Build image lookup
            img_lut = {}
            for root_dir, dirs, files in os.walk(ham_root):
                for f in files:
                    if f.endswith('.jpg'):
                        img_lut[os.path.splitext(f)[0]] = os.path.join(root_dir, f)

            # Split by lesion_id to prevent leakage
            lesion_col = 'lesion_id' if 'lesion_id' in df.columns else None
            if lesion_col:
                lesions = df[lesion_col].unique()
                from sklearn.model_selection import train_test_split
                tr_l, va_l = train_test_split(lesions, test_size=0.15, random_state=42)
                split_lesions = set(tr_l if mode == 'train' else va_l)
                df_split = df[df[lesion_col].isin(split_lesions)]
            else:
                df_split = df

            for _, row in df_split.iterrows():
                dx = str(row.get('dx','')).lower()
                if dx not in SKIN_IDX:
                    continue
                img_id = str(row.get('image_id', row.iloc[0]))
                if img_id in img_lut:
                    self.samples.append((img_lut[img_id], SKIN_IDX[dx]))
            print(f'  HAM10000 [{mode}]: {len(self.samples)} images')

        # ── ISIC 2020 ─────────────────────────────────────────────────────
        if isic_root and os.path.exists(isic_root):
            isic_csv = None
            for r, d, files in os.walk(isic_root):
                for f in files:
                    if f.endswith('.csv') and 'train' in f.lower():
                        isic_csv = os.path.join(r, f)
                        break

            if isic_csv:
                df_isic = pd.read_csv(isic_csv)
                # Map: 0=benign->nv, 1=melanoma->mel
                isic_map = {0: SKIN_IDX.get('nv',5), 1: SKIN_IDX.get('mel',4)}
                train_dir = os.path.join(isic_root, 'train')
                isic_lut = {}
                if os.path.exists(train_dir):
                    for f in os.listdir(train_dir):
                        if f.endswith('.jpg'):
                            isic_lut[os.path.splitext(f)[0]] = os.path.join(train_dir, f)

                n_before = len(self.samples)
                for _, row in df_isic.iterrows():
                    lbl = isic_map.get(int(row.get('target',0)), -1)
                    if lbl == -1:
                        continue
                    img_name = str(row.get('image_name', ''))
                    if img_name in isic_lut:
                        self.samples.append((isic_lut[img_name], lbl))
                print(f'  ISIC 2020 added: {len(self.samples)-n_before} images')

        counts = Counter(s[1] for s in self.samples)
        print(f'  Total skin [{mode}]: {len(self.samples):,}')
        for i, c in enumerate(SKIN_CLASSES):
            print(f'    {c:<6}: {counts.get(i,0):>6}')

    def get_sampler_weights(self):
        labels  = [s[1] for s in self.samples]
        counts  = Counter(labels)
        weights = [1.0 / counts[l] for l in labels]
        return torch.tensor(weights, dtype=torch.double)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        if self.hair_removal:
            img = morphological_hair_removal(img)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


# ── Test ─────────────────────────────────────────────────────────────────
if os.path.exists(HAM_PATH):
    ds_skin = SkinLesionDataset(
        HAM_PATH, isic_root=ISIC_PATH if os.path.exists(ISIC_PATH) else None,
        mode='train', transform=build_skin_transforms('train'), hair_removal=False
    )
    if len(ds_skin) > 0:
        img, lbl = ds_skin[0]
        w = ds_skin.get_sampler_weights()
        print(f'  Sample shape : {img.shape}, label: {lbl} ({SKIN_CLASSES[lbl]})')
        print(f'  Sampler wts  : [{w.min():.6f}, {w.max():.6f}]')
        print('SkinLesionDataset OK')
else:
    print('HAM10000 path not found')

  HAM10000 [train]: 8472 images
  ISIC 2020 added: 0 images
  Total skin [train]: 8,472
    akiec :    275
    bcc   :    420
    bkl   :    919
    df    :     86
    mel   :    937
    nv    :   5722
    vasc  :    113
  Sample shape : torch.Size([3, 224, 224]), label: 2 (bkl)
  Sampler wts  : [0.000175, 0.011628]
SkinLesionDataset OK


---
## Cell 8 - Spine Dataset (RSNA 2024 Lumbar)

In [11]:
import os
import numpy as np
import pandas as pd
import pydicom
import cv2
import torch
from torch.utils.data import Dataset

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')
SPINE_PATH = os.path.join(DS, 'spine', 'rsna_lumbar')

SPINE_CONDITIONS = [
    'spinal_canal_stenosis',
    'left_neural_foraminal_narrowing',
    'right_neural_foraminal_narrowing',
    'left_subarticular_stenosis',
    'right_subarticular_stenosis',
]
SPINE_LEVELS = ['l1_l2','l2_l3','l3_l4','l4_l5','l5_s1']
SEVERITY_MAP = {
    'Normal/Mild': 0, 'normal_mild': 0, 'Normal': 0, 'Mild': 0,
    'Moderate': 1, 'moderate': 1,
    'Severe': 2, 'severe': 2,
}


class SpineDataset(Dataset):
    """
    RSNA 2024 Lumbar Spine dataset.
    25 targets: 5 conditions x 5 levels, each with 3 severity classes.
    Multi-view: Sagittal T1, Sagittal T2/STIR, Axial T2.
    """
    N_TARGETS = 25

    def __init__(self, root, mode='train', img_size=256,
                 val_frac=0.15, seed=42):
        self.root     = root
        self.mode     = mode
        self.img_size = img_size
        self.samples  = []

        train_csv = os.path.join(root, 'train.csv')
        desc_csv  = os.path.join(root, 'train_series_descriptions.csv')

        if not os.path.exists(train_csv):
            print(f'train.csv not found in {root}')
            return

        df_labels = pd.read_csv(train_csv)
        df_desc   = pd.read_csv(desc_csv) if os.path.exists(desc_csv) else None

        # Build target column list (25 targets)
        self.target_cols = []
        for cond in SPINE_CONDITIONS:
            for lvl in SPINE_LEVELS:
                col = f'{cond}_{lvl}'
                if col in df_labels.columns:
                    self.target_cols.append(col)

        print(f'Target columns found: {len(self.target_cols)}/25')

        # Train/val split by study_id
        from sklearn.model_selection import train_test_split
        study_ids = df_labels['study_id'].unique()
        tr_ids, va_ids = train_test_split(study_ids, test_size=val_frac, random_state=seed)
        split_ids = set(tr_ids if mode == 'train' else va_ids)
        df_split = df_labels[df_labels['study_id'].isin(split_ids)].reset_index(drop=True)

        img_dir = os.path.join(root, 'train_images')

        for _, row in df_split.iterrows():
            study_id = str(row['study_id'])
            study_dir = os.path.join(img_dir, study_id)
            if not os.path.exists(study_dir):
                continue

            # Build labels vector
            labels = []
            for col in self.target_cols:
                val = row.get(col, 'Normal/Mild')
                val_str = str(val) if pd.notna(val) else 'Normal/Mild'
                labels.append(SEVERITY_MAP.get(val_str, 0))

            # Pad to 25 if needed
            while len(labels) < self.N_TARGETS:
                labels.append(0)

            # Get series dirs
            series_dirs = [os.path.join(study_dir, s)
                           for s in os.listdir(study_dir)
                           if os.path.isdir(os.path.join(study_dir, s))]

            if series_dirs:
                self.samples.append({
                    'study_id': study_id,
                    'series_dirs': series_dirs,
                    'labels': labels
                })

        print(f'Spine [{mode}]: {len(self.samples)} studies')

    def _load_dicom_series(self, series_dir, n_slices=5):
        """Load n_slices from a DICOM series, return stacked numpy array."""
        dcm_files = sorted([os.path.join(series_dir, f)
                            for f in os.listdir(series_dir)
                            if f.endswith('.dcm')])
        if not dcm_files:
            return np.zeros((n_slices, self.img_size, self.img_size), dtype=np.float32)

        # Sample n_slices evenly
        idxs = np.linspace(0, len(dcm_files)-1, n_slices).astype(int)
        slices = []
        for i in idxs:
            try:
                dcm = pydicom.dcmread(dcm_files[i])
                arr = dcm.pixel_array.astype(np.float32)
                arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
                arr = cv2.resize(arr, (self.img_size, self.img_size))
                slices.append(arr)
            except:
                slices.append(np.zeros((self.img_size, self.img_size), dtype=np.float32))

        return np.stack(slices, axis=0)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        series_dirs = sample['series_dirs']
        labels = torch.tensor(sample['labels'], dtype=torch.long)

        # Load up to 3 series (multi-view: Sag T1, Sag T2, Axial T2)
        views = []
        for sd in series_dirs[:3]:
            view = self._load_dicom_series(sd, n_slices=5)
            views.append(view)

        # Pad to 3 views if fewer series
        while len(views) < 3:
            views.append(np.zeros_like(views[0]))

        # Stack: [3 views x 5 slices, H, W] -> [15, H, W]
        img = np.concatenate(views, axis=0).astype(np.float32)

        if self.mode == 'train' and np.random.rand() > 0.5:
            img = np.flip(img, axis=2).copy()

        return torch.from_numpy(img), labels


# ── Test ─────────────────────────────────────────────────────────────────
if os.path.exists(SPINE_PATH):
    ds_spine = SpineDataset(SPINE_PATH, mode='train', img_size=224)
    if len(ds_spine) > 0:
        img, lbl = ds_spine[0]
        print(f'  Input shape  : {img.shape}  (15 channels = 3 views x 5 slices)')
        print(f'  Labels shape : {lbl.shape}  (25 targets)')
        print(f'  Label values : {lbl[:5].tolist()} ...  (0=Normal/Mild, 1=Moderate, 2=Severe)')
        print('SpineDataset OK')
else:
    print('RSNA Spine path not found')

Target columns found: 25/25
Spine [train]: 1678 studies
  Input shape  : torch.Size([15, 224, 224])  (15 channels = 3 views x 5 slices)
  Labels shape : torch.Size([25])  (25 targets)
  Label values : [0, 0, 0, 0, 0] ...  (0=Normal/Mild, 1=Moderate, 2=Severe)
SpineDataset OK


---
## Cell 9 - Unified DataLoader Factory

In [12]:
import os
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')


def create_dataloaders(
    pipeline,
    batch_size=None,
    num_workers=4,
    img_size=None,
    subset_n=None,
):
    """
    Unified DataLoader factory for all NeuroScope pipelines.

    Args:
        pipeline    : 'brats' | 'brain_cls' | 'utsw' | 'lits' | 'breast' | 'skin' | 'spine'
        batch_size  : override default
        num_workers : DataLoader workers (set 0 on Windows if multiprocessing errors)
        img_size    : override default image size
        subset_n    : limit samples for quick debugging

    Returns:
        (train_loader, val_loader, meta_dict)
    """
    defaults = {
        'brats':     {'batch_size': 1,  'img_size': (96, 96, 96)},
        'brain_cls': {'batch_size': 32, 'img_size': 224},
        'utsw':      {'batch_size': 32, 'img_size': 224},
        'lits':      {'batch_size': 8,  'img_size': 256},
        'breast':    {'batch_size': 16, 'img_size': 512},
        'skin':      {'batch_size': 32, 'img_size': 224},
        'spine':     {'batch_size': 8,  'img_size': 224},
    }
    assert pipeline in defaults, f'Unknown pipeline: {pipeline}. Choose from {list(defaults)}'

    bs  = batch_size or defaults[pipeline]['batch_size']
    isz = img_size   or defaults[pipeline]['img_size']
    sampler = None

    if pipeline == 'brats':
        all_files = get_brats_file_list(os.path.join(DS,'brain','brats2024'))
        if subset_n: all_files = all_files[:subset_n]
        tr_f, va_f = train_test_split(all_files, test_size=0.15, random_state=42)
        tr_ds = BraTSDataset(tr_f, build_brats_transforms('train', isz))
        va_ds = BraTSDataset(va_f, build_brats_transforms('val',   isz))
        meta  = {'num_classes': 4, 'in_channels': 4,
                 'class_names': ['background','NCR','edema','ET'],
                 'label_remap': '4->3 applied in __getitem__'}

    elif pipeline == 'brain_cls':
        tr_ds = BrainClassDataset(os.path.join(DS,'brain','kaggle_brain_tumor'),
                                   'Training', build_brain_cls_transforms('train', isz))
        va_ds = BrainClassDataset(os.path.join(DS,'brain','kaggle_brain_tumor'),
                                   'Testing',  build_brain_cls_transforms('val',   isz))
        if subset_n: tr_ds.samples = tr_ds.samples[:subset_n]
        meta = {'num_classes': 4, 'class_names': BRAIN_CLASSES}

    elif pipeline == 'utsw':
        tfm_tr = build_brain_cls_transforms('train', isz)
        tfm_va = build_brain_cls_transforms('val',   isz)
        tr_ds = UTSWGliomaDataset(os.path.join(DS,'brain','utsw_glioma'),
                                   transform=tfm_tr, split='train')
        va_ds = UTSWGliomaDataset(os.path.join(DS,'brain','utsw_glioma'),
                                   transform=tfm_va, split='val')
        meta = {'num_classes': 3, 'class_names': ['IDH','MGMT','1p19q'],
                'task': 'multi_label_binary'}

    elif pipeline == 'lits':
        nii_pairs = find_lits_pairs(os.path.join(DS,'liver','lits'))
        png_pairs = find_lits_png_pairs(os.path.join(DS,'liver','lits')) if not nii_pairs else []
        pairs = nii_pairs or png_pairs
        if subset_n: pairs = pairs[:subset_n]
        tr_p, va_p = train_test_split(pairs, test_size=0.15, random_state=42)
        tr_ds = LiTSDataset(tr_p, 'train', img_size=isz)
        va_ds = LiTSDataset(va_p, 'val',   img_size=isz)
        meta  = {'num_classes': 3, 'class_names': ['background','liver','tumor']}

    elif pipeline == 'breast':
        tr_ds = CBISDDSMDataset(os.path.join(DS,'breast','cbis_ddsm'),
                                 'train', build_breast_transforms('train', isz))
        va_ds = CBISDDSMDataset(os.path.join(DS,'breast','cbis_ddsm'),
                                 'val',   build_breast_transforms('val',   isz))
        if subset_n: tr_ds.samples = tr_ds.samples[:subset_n]
        meta = {'num_classes': 2, 'class_names': ['benign','malignant']}

    elif pipeline == 'skin':
        tr_ds = SkinLesionDataset(
            os.path.join(DS,'skin','ham10000'),
            isic_root=os.path.join(DS,'skin','isic2020'),
            mode='train', transform=build_skin_transforms('train', isz)
        )
        va_ds = SkinLesionDataset(
            os.path.join(DS,'skin','ham10000'),
            mode='val', transform=build_skin_transforms('val', isz)
        )
        if subset_n:
            tr_ds.samples = tr_ds.samples[:subset_n]
            va_ds.samples = va_ds.samples[:subset_n//5]
        w = tr_ds.get_sampler_weights()
        sampler = WeightedRandomSampler(w, num_samples=len(w), replacement=True)
        meta = {'num_classes': 7, 'class_names': SKIN_CLASSES}

    elif pipeline == 'spine':
        tr_ds = SpineDataset(os.path.join(DS,'spine','rsna_lumbar'), 'train', img_size=isz)
        va_ds = SpineDataset(os.path.join(DS,'spine','rsna_lumbar'), 'val',   img_size=isz)
        if subset_n:
            tr_ds.samples = tr_ds.samples[:subset_n]
            va_ds.samples = va_ds.samples[:subset_n//5]
        meta = {'num_classes': 3, 'n_targets': 25,
                'conditions': SPINE_CONDITIONS, 'levels': SPINE_LEVELS}

    shuffle_tr = sampler is None
    # Note: num_workers > 0 can cause issues on Windows - use 0 if errors occur
    train_loader = DataLoader(tr_ds, batch_size=bs, shuffle=shuffle_tr,
                              sampler=sampler, num_workers=num_workers,
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(va_ds, batch_size=bs, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    meta.update({'train_size': len(tr_ds), 'val_size': len(va_ds), 'batch_size': bs})

    print(f'\n{pipeline} DataLoaders ready')
    print(f'  Train: {len(tr_ds):,} samples -> {len(train_loader):,} batches')
    print(f'  Val  : {len(va_ds):,} samples -> {len(val_loader):,} batches')
    return train_loader, val_loader, meta


print('create_dataloaders() factory defined')
print('Available pipelines: brats | brain_cls | utsw | lits | breast | skin | spine')

create_dataloaders() factory defined
Available pipelines: brats | brain_cls | utsw | lits | breast | skin | spine


---
## Cell 10 - Smoke Test All Pipelines

In [13]:
import os, torch
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
DS   = os.path.join(BASE, 'datasets')

# Test each pipeline with a tiny subset
# num_workers=0 avoids Windows multiprocessing issues
results = {}

pipelines_to_test = [
    ('brain_cls', 16),
    ('utsw',      16),
    ('breast',    16),
    ('skin',      32),
    ('spine',     4),
    ('lits',      4),
]

for pipeline, subset in pipelines_to_test:
    try:
        tl, vl, meta = create_dataloaders(
            pipeline, subset_n=subset, num_workers=0
        )
        imgs, labels = next(iter(tl))
        results[pipeline] = {
            'status': 'OK',
            'img_shape': str(imgs.shape),
            'lbl_shape': str(labels.shape),
            'dtype': str(imgs.dtype)
        }
        print(f'  OK  {pipeline:<12} batch={imgs.shape}  lbl={labels.shape}')
    except Exception as e:
        results[pipeline] = {'status': f'ERROR: {e}'}
        print(f'  ERR {pipeline:<12} {str(e)[:60]}')

# BraTS separately (heavy 3D)
print()
try:
    tl, vl, meta = create_dataloaders('brats', subset_n=2, num_workers=0)
    imgs, labels = next(iter(tl))
    results['brats'] = {'status': 'OK', 'img_shape': str(imgs.shape)}
    print(f'  OK  brats        batch={imgs.shape}  lbl={labels.shape}')
    print(f'       Label unique: {torch.unique(labels).tolist()}  (must not contain 4)')
except Exception as e:
    results['brats'] = {'status': f'ERROR: {e}'}
    print(f'  ERR brats        {str(e)[:60]}')

print()
print('Smoke test summary:')
for k, v in results.items():
    print(f'  {k:<12}: {v["status"]}')

  BrainCls [Training]: 5,600 images
    glioma         :  1400
    meningioma     :  1400
    notumor        :  1400
    pituitary      :  1400
  BrainCls [Testing]: 1,600 images
    glioma         :   400
    meningioma     :   400
    notumor        :   400
    pituitary      :   400

brain_cls DataLoaders ready
  Train: 16 samples -> 0 batches
  Val  : 1,600 samples -> 50 batches
  ERR brain_cls    
UTSW CSV: 30000 rows, 625 patients
UTSW [train]: 25488 slices
UTSW CSV: 30000 rows, 625 patients
UTSW [val]: 4512 slices

utsw DataLoaders ready
  Train: 25,488 samples -> 796 batches
  Val  : 4,512 samples -> 141 batches
  ERR utsw         CLAHE transformation expects 1-channel or 3-channel images.
  CSV matching failed - using all JPEGs with unknown labels
  CBIS-DDSM [train]: 10,237 images
    benign=10,237, malignant=0
  CSV matching failed - using all JPEGs with unknown labels
  CBIS-DDSM [val]: 10,237 images
    benign=10,237, malignant=0

breast DataLoaders ready
  Train: 16 sampl

---
## Cell 11 - Save Preprocessing Module

In [14]:
import os, inspect
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'

# All classes and functions to export
exports = [
    get_brats_file_list, build_brats_transforms, BraTSDataset,
    BRAIN_CLASSES, BRAIN_IDX, build_brain_cls_transforms, BrainClassDataset,
    MARKER_COLS, UTSWGliomaDataset,
    find_lits_pairs, find_lits_png_pairs, LiTSDataset,
    build_breast_transforms, CBISDDSMDataset,
    SKIN_CLASSES, SKIN_IDX, morphological_hair_removal,
    build_skin_transforms, SkinLesionDataset,
    SPINE_CONDITIONS, SPINE_LEVELS, SEVERITY_MAP, SpineDataset,
    create_dataloaders,
]

header = (
    '# NeuroScope AI - neuroscope_datasets.py\n'
    '# Auto-generated by Notebook 02. Do not edit manually.\n'
    'import os, cv2, numpy as np, pandas as pd, torch, nibabel as nib, pydicom\n'
    'import SimpleITK as sitk\n'
    'from pathlib import Path\n'
    'from collections import Counter\n'
    'from sklearn.model_selection import train_test_split\n'
    'from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\n'
    'from monai.transforms import (\n'
    '    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, CropForegroundd,\n'
    '    RandFlipd, RandRotate90d, RandScaleIntensityd, RandShiftIntensityd,\n'
    '    NormalizeIntensityd, EnsureTyped, Orientationd, ResizeWithPadOrCropd)\n'
    'from monai.data import CacheDataset\n'
    'import albumentations as A\n'
    'from albumentations.pytorch import ToTensorV2\n\n'
    'BASE = r\'C:\\Users\\tejan\\OneDrive\\Desktop\\drive\\NeuroScope_AI\'\n'
    'DS   = os.path.join(BASE, \'datasets\')\n\n'
)

blocks = []
for obj in exports:
    if callable(obj) or isinstance(obj, type):
        try:
            blocks.append(inspect.getsource(obj))
        except Exception:
            pass
    else:
        # Constants
        blocks.append(f'{obj.__name__ if hasattr(obj, "__name__") else repr(obj)}')

full = header + '\n\n'.join(blocks)

out_path = os.path.join(BASE, 'neuroscope_datasets.py')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(full)

print(f'Saved: {out_path}')
print(f'Size : {os.path.getsize(out_path)/1024:.1f} KB')
print()
print('Usage in any notebook:')
print("  from neuroscope_datasets import create_dataloaders")
print("  tl, vl, meta = create_dataloaders('brain_cls')")

Saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\neuroscope_datasets.py
Size : 13.8 KB

Usage in any notebook:
  from neuroscope_datasets import create_dataloaders
  tl, vl, meta = create_dataloaders('brain_cls')


---
## Cell 12 - Preprocessing Summary

In [15]:
print('=' * 60)
print('  NEUROSCOPE AI - NOTEBOOK 02 COMPLETE')
print('=' * 60)
print()
print('  Dataset classes:')
print('    BraTSDataset        -- 3D MRI, label remap 4->3')
print('    BrainClassDataset   -- 2D, 4-class, Albumentations')
print('    UTSWGliomaDataset   -- 2D, molecular markers multi-label')
print('    LiTSDataset         -- CT HU windowing, 2D slices')
print('    CBISDDSMDataset     -- Mammography, CLAHE')
print('    SkinLesionDataset   -- Dermoscopy, WeightedRandomSampler')
print('    SpineDataset        -- Multi-view DICOM, 25 targets')
print()
print('  Factory: create_dataloaders(pipeline)')
print('  Module : neuroscope_datasets.py saved')
print()
print('  Next: 03_Brain_Tumor_Segmentation.ipynb')
print('=' * 60)

  NEUROSCOPE AI - NOTEBOOK 02 COMPLETE

  Dataset classes:
    BraTSDataset        -- 3D MRI, label remap 4->3
    BrainClassDataset   -- 2D, 4-class, Albumentations
    UTSWGliomaDataset   -- 2D, molecular markers multi-label
    LiTSDataset         -- CT HU windowing, 2D slices
    CBISDDSMDataset     -- Mammography, CLAHE
    SkinLesionDataset   -- Dermoscopy, WeightedRandomSampler
    SpineDataset        -- Multi-view DICOM, 25 targets

  Factory: create_dataloaders(pipeline)
  Module : neuroscope_datasets.py saved

  Next: 03_Brain_Tumor_Segmentation.ipynb
